# Week 1 progress review

**Project 9: Data re-uploading and the Fourier spectrum of QNNs**  
**This notebook:** data pipeline, fixed folds, classical baselines, first QNN, and report draft.

I created this as a separate notebook for the qnn-fourier-spectrum project and set its kernel metadata to **Python (QNN Fourier Spectrum)**. The result tables stay empty until the code cells run.

## Week 1 checklist

| Requirement | Where I address it |
| --- | --- |
| Data pipeline | Seeded 1D and 2D regression data plus balanced two-spiral classification data |
| Fixed folds | One saved stratified five-fold split reused for every classifier |
| Classical baselines | Fourier-feature linear regression, small MLPs, and Fourier-feature logistic regression |
| First quantum model | Exact statevector one-qubit re-uploading regressor with parameter-shift gradients |
| Report draft | Introduction and methods below |

## How I run the cells

Use **Run All** from top to bottom. The first code cell loads packages and fixes settings. The data cell creates and saves the datasets and folds; the following cells plot the data and fit the classical baselines. The QNN cells define and train the first quantum model, then the spectrum cells check its frequencies. The final code cell saves summaries and model outputs. Later cells depend on variables created earlier, so I need the whole sequence for a complete run.

This checkpoint implements the first layer-count configuration for the 1D QNN. The full 1-6 layer sweep, serial/parallel 2D QNN comparison, and hardware run are later project work.

## Research question, hypothesis, and falsifier

**Research question.** How do the number and placement of data-encoding repetitions determine the Fourier frequencies a QNN can represent?

**Hypothesis.** Repeated Pauli rotations constrain a QNN to a candidate Fourier support set determined by the number of encoding occurrences. Optimization and circuit structure determine which allowed modes receive appreciable coefficients.

**Falsifier.** For the 1D circuit below, a coefficient outside the predicted range \(|k|\leq L\) would count as evidence against the prediction only when its magnitude exceeds \(\max(10^{-6},\,0.01c_{\max})\), where \(c_{\max}\) is the largest coefficient, and the mode persists on a refined grid and across independent initializations. Candidate support and observed coefficients are reported separately.

## Report draft: Introduction

Quantum machine-learning models encode classical inputs into quantum states and use trainable operations to make predictions. In this project, I study data re-uploading, which alternates input-dependent gates with trainable gates so a compact circuit can build richer functions through repeated encoding. Earlier work connects this encoding structure to the Fourier components available to variational quantum models (Pérez-Salinas et al., 2020; Schuld, Sweke, and Meyer, 2021).

I ask how encoding repetitions and circuit depth shape the functions a quantum neural network can represent. My central question is: **How do encoding repetitions and layer count set the frequencies a QNN can represent, and how does the encoding layout affect that representation?** For a feature encoded by \(L\) occurrences of rotations \(R_P(x)=\exp(-ixP/2)\), I expect the candidate integer harmonics to lie in \(\{-L,\ldots,L\}\). This describes possible frequencies; it does not guarantee that training will make every allowed coefficient appear.

I will test the prediction first on controlled periodic regression targets. Then I will compare serial and parallel encodings and apply the models to a balanced two-class spiral dataset. Fourier-feature linear models and small MLPs give me classical reference points. I will report held-out error and Fourier coefficients alongside circuit resources. A coefficient outside the predicted support would count against the prediction only if it clears the predeclared tolerance and remains when I refine the grid and change the initialization.

In this Week 1 notebook, I set up the seeded datasets and fixed classification folds, run the classical baselines, and introduce a one-qubit re-uploading regressor. This gives me a methods and pipeline checkpoint before the full layer sweep and hardware evaluation.

**References.** A. Pérez-Salinas et al., *Quantum* 4, 226 (2020), doi:10.22331/q-2020-02-06-226. M. Schuld, R. Sweke, and J. J. Meyer, *Phys. Rev. A* 103, 032430 (2021), doi:10.1103/PhysRevA.103.032430.

## Report draft: Methods

**Synthetic regression.** I sample inputs uniformly from \([-\pi,\pi)\) using root data seed 42. My 1D target is \(f_1(x)=0.5\cos x+0.3\sin(2x)+0.2\cos(3x)\); I use 128 training points and a dense 256-point evaluation grid. My 2D target is \(f_2(x_1,x_2)=0.4\sin x_1+0.3\cos(2x_2)+0.3\sin(x_1-x_2)\); I use 2,048 training points and a \(64\times64\) evaluation grid. I derive separate deterministic random streams from the same root seed.

**Classification and folds.** I generate a balanced two-spiral dataset with 1,000 examples per class. I create one shuffled, stratified five-fold split with seed 42, check that every example appears in exactly one test fold, save the indices, and reuse those folds for every classifier. I follow the NB11 harness pattern: any learned scaling stays inside the estimator pipeline and is fitted on that fold's training data only. I report accuracy and macro F1 for each fold, then give the mean and sample standard deviation.

**Classical baselines.** I compare Fourier-feature linear regression with small MLPs. For 1D regression, my MLP has 7 trainable weights and biases, close to the QNN's 6 trainable rotation angles; the Fourier model has 6 coefficients plus an intercept. For 2D regression, the MLP and Fourier model each have 49 coefficients including the intercept. For the spiral task, I use Fourier-feature logistic regression with integer modes through 4 and a small 2-5-1 MLP. I fix these settings before evaluation and do not tune them on the reported folds.

**First QNN.** I start with a noiseless one-qubit statevector regressor with \(L=3\) re-uploading layers. Each layer applies \(R_Y(x)\), followed by trainable \(R_Z(\alpha_\ell)\) and \(R_Y(\beta_\ell)\); I use the expectation of \(Z\) as the prediction. The model has 6 trainable angles. I train with full-batch Adam at learning rate 0.01 for up to 1,000 updates and use initialization seeds 0-4. I calculate parameter-shift gradients for the trainable Pauli rotations. I evaluate grid MSE and the discrete Fourier transform on 256 points, then use a 512-point grid to check that the spectrum is stable. This NumPy statevector exactly simulates the stated one-qubit circuit; it is not a hardware run.

**Analysis and reproducibility.** I save the generated inputs, targets, fold indices, model summaries, QNN parameters, predictions, and spectrum checks under outputs/week1. I also record package versions, model settings, seeds, and fold definitions so I can reproduce the tables and figures.

## 1. Setup and locked configuration

I run this notebook from the project root with the Python (QNN Fourier Spectrum) kernel. The first code cell checks for the required packages and, if any are missing, installs the dependencies from requirements.txt using the Python executable behind that selected kernel. Package installation needs access to the configured package index.

In [ ]:
from pathlib import Path
import importlib.util
import json
import platform
from importlib import metadata
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "synopsis.tex").exists():
    raise FileNotFoundError(
        "Start Jupyter from qnn-fourier-spectrum or its notebooks directory."
    )

REQUIREMENTS_FILE = PROJECT_ROOT / "requirements.txt"
if not REQUIREMENTS_FILE.exists():
    raise FileNotFoundError(f"Could not find project requirements: {REQUIREMENTS_FILE}")

required_modules = ("numpy", "pandas", "matplotlib", "IPython", "sklearn")
missing_modules = [
    module_name
    for module_name in required_modules
    if importlib.util.find_spec(module_name) is None
]

if missing_modules:
    print("Missing packages:", ", ".join(missing_modules))
    print("Installing requirements into the selected kernel:", sys.executable)
    try:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(REQUIREMENTS_FILE),
            ]
        )
    except (subprocess.CalledProcessError, FileNotFoundError) as error:
        raise RuntimeError(
            "Package installation failed. Confirm that this kernel's Python has pip "
            "and can reach the configured package index, then install requirements.txt."
        ) from error
else:
    print("Required packages are already available in the selected kernel.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEED = 42
N_FOLDS = 5
N_REG_1D = 128
N_REG_2D = 2048
N_SPIRAL_PER_CLASS = 1000
N_QNN_LAYERS = 3
QNN_INIT_SEEDS = tuple(range(5))
QNN_MAX_STEPS = 1000
QNN_LEARNING_RATE = 0.01
REGRESSION_HARMONIC_CUTOFF = 3
SPIRAL_HARMONIC_CUTOFF = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "week1"

print("Project root:", PROJECT_ROOT)
print("Kernel executable:", sys.executable)
print("Python:", platform.python_version())
for distribution in ("numpy", "pandas", "matplotlib", "ipython", "scikit-learn"):
    print(f"{distribution}: {metadata.version(distribution)}")

## 2. Seeded data pipeline and fixed folds

I derive the data streams from one recorded seed. The regression evaluation grids stay separate from the sampled training data. I build the spiral folds once here, check their coverage, save the indices, and reuse them below.

In [ ]:
def target_1d(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return 0.5 * np.cos(x) + 0.3 * np.sin(2 * x) + 0.2 * np.cos(3 * x)


def target_2d(X):
    X = np.asarray(X, dtype=float)
    x1, x2 = X[:, 0], X[:, 1]
    return 0.4 * np.sin(x1) + 0.3 * np.cos(2 * x2) + 0.3 * np.sin(x1 - x2)


def make_two_spirals(n_per_class, rng, noise_sd=0.025):
    radius = np.linspace(0.02, 1.0, n_per_class)
    angle = np.linspace(0.0, 4.0 * np.pi, n_per_class, endpoint=False)
    arm_0 = np.column_stack((radius * np.cos(angle), radius * np.sin(angle)))
    arm_1 = np.column_stack(
        (radius * np.cos(angle + np.pi), radius * np.sin(angle + np.pi))
    )
    X = np.vstack((arm_0, arm_1))
    y = np.concatenate(
        (np.zeros(n_per_class, dtype=int), np.ones(n_per_class, dtype=int))
    )
    X += rng.normal(0.0, noise_sd, size=X.shape)
    order = rng.permutation(len(y))
    return X[order], y[order]


rng_1d, rng_2d, rng_spiral = [
    np.random.default_rng(child)
    for child in np.random.SeedSequence(SEED).spawn(3)
]

X_1d_train = rng_1d.uniform(-np.pi, np.pi, size=(N_REG_1D, 1))
y_1d_train = target_1d(X_1d_train[:, 0])
x_1d_grid = np.linspace(-np.pi, np.pi, 256, endpoint=False)
X_1d_grid = x_1d_grid.reshape(-1, 1)
y_1d_grid = target_1d(x_1d_grid)

X_2d_train = rng_2d.uniform(-np.pi, np.pi, size=(N_REG_2D, 2))
y_2d_train = target_2d(X_2d_train)
axis_2d = np.linspace(-np.pi, np.pi, 64, endpoint=False)
grid_x1, grid_x2 = np.meshgrid(axis_2d, axis_2d, indexing="xy")
X_2d_grid = np.column_stack((grid_x1.ravel(), grid_x2.ravel()))
y_2d_grid = target_2d(X_2d_grid).reshape(64, 64)

X_spiral, y_spiral = make_two_spirals(
    N_SPIRAL_PER_CLASS, rng_spiral
)

splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FIXED_FOLDS = [
    (train_idx.astype(int), test_idx.astype(int))
    for train_idx, test_idx in splitter.split(X_spiral, y_spiral)
]
FOLD_TRAIN = np.stack([train_idx for train_idx, _ in FIXED_FOLDS])
FOLD_TEST = np.stack([test_idx for _, test_idx in FIXED_FOLDS])

test_coverage = np.zeros(len(y_spiral), dtype=int)
for train_idx, test_idx in FIXED_FOLDS:
    assert len(np.intersect1d(train_idx, test_idx)) == 0
    test_coverage[test_idx] += 1
assert np.all(test_coverage == 1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUTPUT_DIR / "week1_inputs_and_fixed_folds.npz",
    X_1d_train=X_1d_train,
    y_1d_train=y_1d_train,
    X_1d_grid=X_1d_grid,
    y_1d_grid=y_1d_grid,
    X_2d_train=X_2d_train,
    y_2d_train=y_2d_train,
    X_2d_grid=X_2d_grid,
    y_2d_grid=y_2d_grid,
    X_spiral=X_spiral,
    y_spiral=y_spiral,
    fold_train=FOLD_TRAIN,
    fold_test=FOLD_TEST,
)

print("1D regression:", X_1d_train.shape, "| dense grid:", X_1d_grid.shape)
print("2D regression:", X_2d_train.shape, "| dense grid:", y_2d_grid.shape)
print("Spirals:", X_spiral.shape, "| class counts:", np.bincount(y_spiral))
print("Fixed folds:", len(FIXED_FOLDS), "| train/test:", FOLD_TRAIN.shape[1], FOLD_TEST.shape[1])
print("Saved inputs and fold indices:", OUTPUT_DIR / "week1_inputs_and_fixed_folds.npz")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X_1d_train[:, 0], y_1d_train, s=12, alpha=0.7)
axes[0].plot(x_1d_grid, y_1d_grid, color="black", linewidth=1.5)
axes[0].set(title="1D periodic regression", xlabel="x", ylabel="f1(x)")

image_2d = axes[1].contourf(
    grid_x1, grid_x2, y_2d_grid, levels=20, cmap="coolwarm"
)
axes[1].set(title="2D periodic target", xlabel="x1", ylabel="x2")
fig.colorbar(image_2d, ax=axes[1], shrink=0.8)

for label in (0, 1):
    mask = y_spiral == label
    axes[2].scatter(
        X_spiral[mask, 0], X_spiral[mask, 1], s=5, alpha=0.65, label=f"class {label}"
    )
axes[2].set(title="Balanced two-spiral data", xlabel="x1", ylabel="x2", aspect="equal")
axes[2].legend()
fig.tight_layout()

## 3. Classical regression baselines

I use integer Fourier modes within each target's declared support. The 1D MLP has 7 trainable parameters, close to the QNN's 6 angles. The 2D MLP and Fourier linear model each have 49 coefficients including the intercept.

In [ ]:
def fourier_features_1d(X, max_harmonic=3):
    x = np.asarray(X, dtype=float).reshape(-1)
    columns = []
    for k in range(1, max_harmonic + 1):
        columns.extend((np.cos(k * x), np.sin(k * x)))
    return np.column_stack(columns)


def integer_modes_2d(max_harmonic):
    modes = []
    for k1 in range(-max_harmonic, max_harmonic + 1):
        for k2 in range(-max_harmonic, max_harmonic + 1):
            if k1 == 0 and k2 == 0:
                continue
            if k1 < 0 or (k1 == 0 and k2 < 0):
                continue
            modes.append((k1, k2))
    return np.asarray(modes, dtype=float)


def fourier_features_2d(X, max_harmonic=3, angular_scale=1.0):
    X = np.asarray(X, dtype=float)
    modes = integer_modes_2d(max_harmonic)
    phase = angular_scale * (X @ modes.T)
    return np.column_stack((np.cos(phase), np.sin(phase)))


fourier_1d = LinearRegression().fit(
    fourier_features_1d(X_1d_train, REGRESSION_HARMONIC_CUTOFF),
    y_1d_train,
)
mlp_1d = MLPRegressor(
    hidden_layer_sizes=(2,),
    activation="tanh",
    solver="adam",
    learning_rate_init=0.01,
    max_iter=4000,
    tol=1e-7,
    n_iter_no_change=250,
    random_state=SEED,
).fit(X_1d_train, y_1d_train)

fourier_2d = LinearRegression().fit(
    fourier_features_2d(X_2d_train, REGRESSION_HARMONIC_CUTOFF),
    y_2d_train,
)
mlp_2d = MLPRegressor(
    hidden_layer_sizes=(12,),
    activation="tanh",
    solver="adam",
    learning_rate_init=0.01,
    max_iter=4000,
    tol=1e-7,
    n_iter_no_change=250,
    random_state=SEED,
).fit(X_2d_train, y_2d_train)

predictions_1d = {
    "Fourier linear, K<=3": fourier_1d.predict(
        fourier_features_1d(X_1d_grid, REGRESSION_HARMONIC_CUTOFF)
    ),
    "MLP 1-2-1": mlp_1d.predict(X_1d_grid),
}
predictions_2d = {
    "Fourier linear, |k_j|<=3": fourier_2d.predict(
        fourier_features_2d(X_2d_grid, REGRESSION_HARMONIC_CUTOFF)
    ),
    "MLP 2-12-1": mlp_2d.predict(X_2d_grid),
}

regression_rows = [
    {
        "task": "1D",
        "model": "Fourier linear, K<=3",
        "trainable_parameters": 7,
        "dense_grid_mse": mean_squared_error(y_1d_grid, predictions_1d["Fourier linear, K<=3"]),
    },
    {
        "task": "1D",
        "model": "MLP 1-2-1",
        "trainable_parameters": 7,
        "dense_grid_mse": mean_squared_error(y_1d_grid, predictions_1d["MLP 1-2-1"]),
    },
    {
        "task": "2D",
        "model": "Fourier linear, |k_j|<=3",
        "trainable_parameters": 49,
        "dense_grid_mse": mean_squared_error(y_2d_grid.ravel(), predictions_2d["Fourier linear, |k_j|<=3"]),
    },
    {
        "task": "2D",
        "model": "MLP 2-12-1",
        "trainable_parameters": 49,
        "dense_grid_mse": mean_squared_error(y_2d_grid.ravel(), predictions_2d["MLP 2-12-1"]),
    },
]
regression_summary = pd.DataFrame(regression_rows)
regression_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_1d_grid, y_1d_grid, color="black", linewidth=2, label="target")
for name, prediction in predictions_1d.items():
    axes[0].plot(x_1d_grid, prediction, label=name)
axes[0].set(title="1D baseline fits", xlabel="x", ylabel="prediction")
axes[0].legend()

for name, prediction in predictions_2d.items():
    axes[1].plot(
        y_2d_grid.ravel()[::32],
        prediction[::32],
        ".",
        alpha=0.65,
        label=name,
    )
axes[1].plot(y_2d_grid.ravel()[::32], y_2d_grid.ravel()[::32], color="black", label="target")
axes[1].set(title="2D target vs. predictions", xlabel="target", ylabel="prediction")
axes[1].legend()
fig.tight_layout()

## 4. Spiral classification baselines on the frozen folds

I keep the Fourier map fixed from the declared coordinate domain. The MLP learns its standardization inside each fold's pipeline. I reuse the same fold indices for both models; this follows the NB11 harness's fixed-fold and training-only preprocessing approach.

In [ ]:
def spiral_fourier_features(X):
    return fourier_features_2d(
        X,
        max_harmonic=SPIRAL_HARMONIC_CUTOFF,
        angular_scale=np.pi,
    )


spiral_models = {
    "Fourier logistic, K<=4": Pipeline(
        [
            ("features", FunctionTransformer(spiral_fourier_features, validate=False)),
            ("classifier", LogisticRegression(C=1.0, max_iter=3000, random_state=SEED)),
        ]
    ),
    "MLP 2-5-1": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "classifier",
                MLPClassifier(
                    hidden_layer_sizes=(5,),
                    activation="tanh",
                    solver="adam",
                    learning_rate_init=0.01,
                    max_iter=2500,
                    tol=1e-5,
                    n_iter_no_change=150,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}

SPIRAL_PARAMETER_COUNTS = {
    "Fourier logistic, K<=4": 2 * len(integer_modes_2d(SPIRAL_HARMONIC_CUTOFF)) + 1,
    "MLP 2-5-1": 21,
}
classification_rows = []
classification_fold_scores = {}
for model_name, estimator in spiral_models.items():
    scores = cross_validate(
        estimator,
        X_spiral,
        y_spiral,
        cv=FIXED_FOLDS,
        scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
        return_train_score=False,
        n_jobs=1,
        error_score="raise",
    )
    classification_fold_scores[model_name] = scores
    classification_rows.append(
        {
            "model": model_name,
            "trainable_parameters": SPIRAL_PARAMETER_COUNTS[model_name],
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_sd": scores["test_accuracy"].std(ddof=1),
            "macro_f1_mean": scores["test_macro_f1"].mean(),
            "macro_f1_sd": scores["test_macro_f1"].std(ddof=1),
            "fixed_folds": N_FOLDS,
        }
    )

classification_fold_rows = []
for model_name, scores in classification_fold_scores.items():
    for fold_index in range(N_FOLDS):
        classification_fold_rows.append(
            {
                "model": model_name,
                "fold": fold_index + 1,
                "accuracy": scores["test_accuracy"][fold_index],
                "macro_f1": scores["test_macro_f1"][fold_index],
            }
        )
classification_fold_scores_table = pd.DataFrame(classification_fold_rows)
classification_summary = pd.DataFrame(classification_rows)
classification_summary

## 5. First quantum model: one-qubit re-uploading regressor

I use an exact statevector simulator for the circuit described in the methods draft. Each trainable rotation angle occurs once, so I can apply the standard two-term parameter-shift rule directly.

In [ ]:
def ry_matrix(angle):
    cosine = np.cos(angle / 2.0)
    sine = np.sin(angle / 2.0)
    return np.array([[cosine, -sine], [sine, cosine]], dtype=complex)


def rz_matrix(angle):
    return np.diag(
        [np.exp(-0.5j * angle), np.exp(0.5j * angle)]
    ).astype(complex)


def qnn_1d_expectation(x, theta, n_layers=N_QNN_LAYERS):
    x = np.asarray(x, dtype=float).reshape(-1)
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if theta.size != 2 * n_layers:
        raise ValueError(f"Expected {2 * n_layers} trainable angles, got {theta.size}.")

    state = np.zeros((len(x), 2), dtype=complex)
    state[:, 0] = 1.0
    cosine = np.cos(x / 2.0)
    sine = np.sin(x / 2.0)

    for layer in range(n_layers):
        input_encoded = np.empty_like(state)
        input_encoded[:, 0] = cosine * state[:, 0] - sine * state[:, 1]
        input_encoded[:, 1] = sine * state[:, 0] + cosine * state[:, 1]
        state = input_encoded
        state = state @ rz_matrix(theta[2 * layer]).T
        state = state @ ry_matrix(theta[2 * layer + 1]).T

    probability_zero = np.abs(state[:, 0]) ** 2
    probability_one = np.abs(state[:, 1]) ** 2
    return probability_zero - probability_one


def qnn_parameter_shift_jacobian(x, theta, n_layers=N_QNN_LAYERS):
    theta = np.asarray(theta, dtype=float)
    jacobian = np.empty((len(np.asarray(x).reshape(-1)), theta.size), dtype=float)
    for parameter_index in range(theta.size):
        plus = theta.copy()
        minus = theta.copy()
        plus[parameter_index] += np.pi / 2.0
        minus[parameter_index] -= np.pi / 2.0
        jacobian[:, parameter_index] = 0.5 * (
            qnn_1d_expectation(x, plus, n_layers)
            - qnn_1d_expectation(x, minus, n_layers)
        )
    return jacobian

In [ ]:
def fit_qnn_1d(
    X,
    y,
    n_layers=N_QNN_LAYERS,
    seed=0,
    max_steps=QNN_MAX_STEPS,
    learning_rate=QNN_LEARNING_RATE,
):
    x = np.asarray(X, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)
    rng = np.random.default_rng(seed)
    theta = rng.uniform(-np.pi, np.pi, size=2 * n_layers)
    first_moment = np.zeros_like(theta)
    second_moment = np.zeros_like(theta)
    loss_history = []

    for step in range(1, max_steps + 1):
        prediction = qnn_1d_expectation(x, theta, n_layers)
        residual = prediction - y
        loss = float(np.mean(residual ** 2))
        jacobian = qnn_parameter_shift_jacobian(x, theta, n_layers)
        gradient = (2.0 / len(x)) * (residual @ jacobian)

        first_moment = 0.9 * first_moment + 0.1 * gradient
        second_moment = 0.999 * second_moment + 0.001 * (gradient ** 2)
        corrected_first = first_moment / (1.0 - 0.9 ** step)
        corrected_second = second_moment / (1.0 - 0.999 ** step)
        theta -= learning_rate * corrected_first / (np.sqrt(corrected_second) + 1e-8)

        if step == 1 or step % 50 == 0 or step == max_steps:
            loss_history.append({"step": step, "train_mse": loss})

    return theta, pd.DataFrame(loss_history)


qnn_models = {}
qnn_prediction_rows = []
qnn_train_histories = {}
for init_seed in QNN_INIT_SEEDS:
    fitted_theta, history = fit_qnn_1d(
        X_1d_train,
        y_1d_train,
        n_layers=N_QNN_LAYERS,
        seed=init_seed,
    )
    qnn_models[init_seed] = fitted_theta
    qnn_train_histories[init_seed] = history
    train_prediction = qnn_1d_expectation(X_1d_train, fitted_theta)
    grid_prediction = qnn_1d_expectation(X_1d_grid, fitted_theta)
    qnn_prediction_rows.append(
        {
            "initialization_seed": init_seed,
            "train_mse": mean_squared_error(y_1d_train, train_prediction),
            "dense_grid_mse": mean_squared_error(y_1d_grid, grid_prediction),
            "layers": N_QNN_LAYERS,
            "trainable_angles": len(fitted_theta),
            "adam_updates": QNN_MAX_STEPS,
        }
    )

qnn_regression_runs = pd.DataFrame(qnn_prediction_rows)
qnn_regression_runs

In [ ]:
qnn_grid_predictions = np.stack(
    [
        qnn_1d_expectation(X_1d_grid, qnn_models[seed])
        for seed in QNN_INIT_SEEDS
    ]
)
qnn_grid_mean = qnn_grid_predictions.mean(axis=0)
qnn_grid_sd = qnn_grid_predictions.std(axis=0, ddof=1)

qnn_summary = pd.DataFrame(
    [
        {
            "task": "1D",
            "model": f"QNN re-uploading, L={N_QNN_LAYERS}",
            "trainable_parameters": 2 * N_QNN_LAYERS,
            "dense_grid_mse_mean": qnn_regression_runs["dense_grid_mse"].mean(),
            "dense_grid_mse_sd": qnn_regression_runs["dense_grid_mse"].std(ddof=1),
            "initializations": len(QNN_INIT_SEEDS),
        }
    ]
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_1d_grid, y_1d_grid, color="black", linewidth=2, label="target")
ax.plot(
    x_1d_grid,
    predictions_1d["Fourier linear, K<=3"],
    linestyle="--",
    label="Fourier linear, K<=3",
)
ax.plot(x_1d_grid, predictions_1d["MLP 1-2-1"], linestyle=":", label="MLP 1-2-1")
ax.plot(x_1d_grid, qnn_grid_mean, label=f"QNN mean, L={N_QNN_LAYERS}")
ax.fill_between(
    x_1d_grid,
    qnn_grid_mean - qnn_grid_sd,
    qnn_grid_mean + qnn_grid_sd,
    alpha=0.18,
    label="QNN +/- 1 initialization SD",
)
ax.set(title="Week 1 1D regression comparison", xlabel="x", ylabel="prediction")
ax.legend(ncol=2)
fig.tight_layout()

display(pd.concat([regression_summary.query("task == '1D'"), qnn_summary], ignore_index=True))

## 6. Fourier-spectrum check and grid refinement

On the periodic grid, the DFT bins correspond to integer harmonics. For each initialization, I check the largest coefficient outside \(|k|\leq L\) at 256 and 512 points and apply the predeclared threshold. I treat this as a diagnostic; a single numerical result is not enough to claim that the prediction failed.

In [ ]:
def shifted_dft(values):
    values = np.asarray(values, dtype=float).reshape(-1)
    n_points = len(values)
    frequencies = np.rint(
        np.fft.fftshift(np.fft.fftfreq(n_points, d=1.0 / n_points))
    ).astype(int)
    coefficients = np.fft.fftshift(np.fft.fft(values) / n_points)
    return frequencies, coefficients


spectrum_rows = []
spectrum_magnitudes = []
fine_x = np.linspace(-np.pi, np.pi, 512, endpoint=False).reshape(-1, 1)

for init_seed in QNN_INIT_SEEDS:
    for n_points, evaluation_grid in ((256, X_1d_grid), (512, fine_x)):
        prediction = qnn_1d_expectation(evaluation_grid, qnn_models[init_seed])
        frequencies, coefficients = shifted_dft(prediction)
        magnitudes = np.abs(coefficients)
        threshold = max(1e-6, 0.01 * float(magnitudes.max()))
        outside = np.abs(frequencies) > N_QNN_LAYERS
        above_threshold = outside & (magnitudes > threshold)
        outside_frequency = (
            frequencies[above_threshold][np.argmax(magnitudes[above_threshold])]
            if np.any(above_threshold)
            else 0
        )
        outside_magnitude = (
            float(magnitudes[above_threshold].max())
            if np.any(above_threshold)
            else 0.0
        )
        spectrum_rows.append(
            {
                "initialization_seed": init_seed,
                "grid_points": n_points,
                "largest_coefficient": float(magnitudes.max()),
                "threshold": threshold,
                "largest_outside_frequency": int(outside_frequency),
                "largest_outside_magnitude": outside_magnitude,
                "outside_mode_above_threshold": bool(np.any(above_threshold)),
            }
        )
        if n_points == 256:
            spectrum_magnitudes.append(magnitudes)

spectrum_summary = pd.DataFrame(spectrum_rows)
spectrum_summary

In [ ]:
target_frequencies, target_coefficients = shifted_dft(y_1d_grid)
mean_qnn_magnitude = np.mean(np.stack(spectrum_magnitudes), axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.stem(
    target_frequencies,
    np.abs(target_coefficients),
    linefmt="C0-",
    markerfmt="C0o",
    basefmt=" ",
    label="target",
)
ax.stem(
    target_frequencies,
    mean_qnn_magnitude,
    linefmt="C1-",
    markerfmt="C1s",
    basefmt=" ",
    label="mean QNN coefficient magnitude",
)
ax.axvline(N_QNN_LAYERS + 0.5, color="black", linestyle="--", linewidth=1)
ax.axvline(-N_QNN_LAYERS - 0.5, color="black", linestyle="--", linewidth=1)
ax.set(
    xlim=(-12, 12),
    xlabel="integer Fourier frequency k",
    ylabel="coefficient magnitude",
    title="Target and first QNN spectrum (candidate support: |k| <= L)",
)
ax.legend()
fig.tight_layout()

print("Candidate support:", list(range(-N_QNN_LAYERS, N_QNN_LAYERS + 1)))
display(spectrum_summary)

## 7. Save Week 1 summaries and reproducibility metadata

I create these result files by running the cells above. The config records that this checkpoint evaluates one 1D layer count, not the full planned sweep.

In [ ]:
regression_summary.to_csv(OUTPUT_DIR / "regression_baselines.csv", index=False)
qnn_regression_runs.to_csv(OUTPUT_DIR / "qnn_1d_initializations.csv", index=False)
classification_summary.to_csv(OUTPUT_DIR / "spiral_classification_baselines.csv", index=False)
classification_fold_scores_table.to_csv(OUTPUT_DIR / "spiral_classification_fold_scores.csv", index=False)
spectrum_summary.to_csv(OUTPUT_DIR / "qnn_1d_spectrum_check.csv", index=False)

np.savez_compressed(
    OUTPUT_DIR / "qnn_1d_predictions_and_parameters.npz",
    x_grid=x_1d_grid,
    target=y_1d_grid,
    predictions=qnn_grid_predictions,
    initialization_seeds=np.asarray(QNN_INIT_SEEDS),
    parameters=np.stack([qnn_models[seed] for seed in QNN_INIT_SEEDS]),
)
np.savez_compressed(
    OUTPUT_DIR / "classical_regression_predictions.npz",
    x_1d_grid=x_1d_grid,
    y_1d_grid=y_1d_grid,
    fourier_1d=predictions_1d["Fourier linear, K<=3"],
    mlp_1d=predictions_1d["MLP 1-2-1"],
    X_2d_grid=X_2d_grid,
    y_2d_grid=y_2d_grid,
    fourier_2d=predictions_2d["Fourier linear, |k_j|<=3"],
    mlp_2d=predictions_2d["MLP 2-12-1"],
)

package_versions = {}
for distribution in ("numpy", "pandas", "matplotlib", "scikit-learn"):
    try:
        package_versions[distribution] = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        package_versions[distribution] = "not installed"

run_config = {
    "project": "Project 9: Data re-uploading and the Fourier spectrum of QNNs",
    "week": 1,
    "root_seed": SEED,
    "regression": {
        "n_1d_train": N_REG_1D,
        "n_1d_grid": len(X_1d_grid),
        "n_2d_train": N_REG_2D,
        "n_2d_grid_per_axis": 64,
        "target_fourier_cutoff": REGRESSION_HARMONIC_CUTOFF,
    },
    "classification": {
        "dataset": "balanced synthetic two spirals",
        "examples_per_class": N_SPIRAL_PER_CLASS,
        "folds": N_FOLDS,
        "fold_seed": SEED,
    },
    "first_qnn": {
        "simulator": "exact NumPy statevector",
        "qubits": 1,
        "layers": N_QNN_LAYERS,
        "trainable_angles": 2 * N_QNN_LAYERS,
        "initialization_seeds": list(QNN_INIT_SEEDS),
        "optimizer": "full-batch Adam",
        "learning_rate": QNN_LEARNING_RATE,
        "maximum_updates": QNN_MAX_STEPS,
    },
    "python_version": platform.python_version(),
    "package_versions": package_versions,
}
with (OUTPUT_DIR / "week1_run_config.json").open("w", encoding="utf-8") as stream:
    json.dump(run_config, stream, indent=2)

print("Saved Week 1 artifacts to:", OUTPUT_DIR)
print("Files: regression_baselines.csv, qnn_1d_initializations.csv,")
print("       spiral_classification_baselines.csv, qnn_1d_spectrum_check.csv,")
print("       spiral_classification_fold_scores.csv, classical_regression_predictions.npz,")
print("       qnn_1d_predictions_and_parameters.npz, week1_run_config.json")
display(regression_summary)
display(qnn_regression_runs)
display(classification_summary)

## Week 1 interpretation and next steps

After I run all cells in the project kernel, I can add the observed baseline and QNN summaries to the report draft. I will report all five initialization runs and their spread rather than choosing a favorable run after seeing the scores.

Next, I will repeat the regression for \(L=1,\ldots,6\), extend the model to 2D serial and parallel encodings, and report their resource counts. I will add the two-qubit spiral QNN before making matched classification comparisons. Any hardware result will be reported separately from this noiseless simulation.